Created Train/Test split with the latest month (September 2026) being the test split and the year before being the train split.

In [150]:
from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

TEST_MONTH = "202609"
TRAIN_MONTHS = [
    "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604",
    "202605", "202606", "202607", "202608",
]

In [151]:
def sold_path(month):
    generated = CSV_DIR / f"GeneratedCRMLSSold{month}.csv"
    raw = CSV_DIR / f"CRMLSSold{month}.csv"
    if generated.exists():
        return generated
    return raw


def load_month(month):
    df = pd.read_csv(sold_path(month), low_memory=False)
    df["month"] = month
    return df


train = pd.concat([load_month(m) for m in TRAIN_MONTHS], ignore_index=True)
test = load_month(TEST_MONTH)

print("train months", TRAIN_MONTHS)
print(train["month"].value_counts().sort_index())
print(test["month"].value_counts().sort_index())

train months ['202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606', '202607', '202608']
month
202509    22443
202510    23233
202511    19088
202512    20538
202601    16487
202602    19010
202603    23372
202604    24261
202605    24277
202606    25812
202607    25467
202608    23133
Name: count, dtype: int64
month
202609    22107
Name: count, dtype: int64


Combining test/train tables into one in order to handle null values and encode categorical columns. 

In [152]:
df = pd.concat(
    [train.assign(split="train"), test.assign(split="test")],
    ignore_index=True,
)
df["split"].value_counts()

split
train    267121
test      22107
Name: count, dtype: int64

Keep the home features used for modeling, columns relating to listing itself are not included

In [153]:
keep_cols = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "PropertyType",
    "PropertySubType",
    "WaterfrontYN",
    "BasementYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
    "City",
    "PostalCode",
    "Latitude",
    "Longitude",
    "month",
    "Flooring",
    "FireplacesTotal",
    "AssociationFeeFrequency",
    "CountyOrParish",
    "ParkingTotal",
    "PropertySubType",
    "LotSizeAcres",
    "BuildingAreaTotal",
    "ElementarySchoolDistrict",
    "StateOrProvince",
    "CoveredSpaces",
    "Stories",
    "Levels",
    "HighSchoolDistrict",
    "AssociationFee",
    "MiddleOrJuniorSchoolDistrict",
    "split",
] 
df = df[keep_cols]

Determining the proportion of entries that have a null value for each column

In [154]:
null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
null_pct

MiddleOrJuniorSchoolDistrict    100.000000
CoveredSpaces                   100.000000
ElementarySchoolDistrict        100.000000
FireplacesTotal                 100.000000
WaterfrontYN                     99.943989
BasementYN                       98.363229
BuildingAreaTotal                86.603303
AssociationFeeFrequency          70.411578
Flooring                         41.415423
HighSchoolDistrict               34.165433
AssociationFee                   30.470079
AttachedGarageYN                 28.433969
Stories                          17.907671
GarageSpaces                     12.800974
NewConstructionYN                12.000221
Levels                           11.499924
PoolPrivateYN                    10.729252
ViewYN                           10.268370
LotSizeAcres                      8.551385
FireplaceYN                       8.336330
LotSizeSquareFeet                 8.296915
PropertySubType                   7.141771
PropertySubType                   7.141771
LivingArea 

Drop columns that are null for every row, observe remaining columns

In [155]:
all_null = df.columns[df.isna().all()]
df = df.drop(columns=all_null)

# Dropping null Close Price rows because they are not useful for modeling,
# Dropping null State/Province, Latitude, and Longitude rows because of low null percentages
df = df.dropna(subset=["ClosePrice", "StateOrProvince", "Latitude", "Longitude"]) 

null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())


WaterfrontYN               99.943579
BasementYN                 98.357870
BuildingAreaTotal          86.521644
AssociationFeeFrequency    70.400206
Flooring                   41.326660
HighSchoolDistrict         33.903116
AssociationFee             30.493823
AttachedGarageYN           28.289224
Stories                    17.684161
GarageSpaces               12.699955
NewConstructionYN          11.758560
Levels                     11.307192
PoolPrivateYN              10.513118
ViewYN                     10.161357
LotSizeAcres                8.332550
FireplaceYN                 8.154928
LotSizeSquareFeet           8.076217
PropertySubType             6.975311
PropertySubType             6.975311
LivingArea                  6.654198
BedroomsTotal               6.213627
BathroomsTotalInteger       4.385516
YearBuilt                   3.909420
ParkingTotal                2.794930
PostalCode                  0.027514
City                        0.009055


See the values a column takes, including null

For waterfront column, notice that the only non-null value is true meaning we can impute the null values to be false and then encode the columns by making true = 1 and false = 0

In [156]:
def colValues(col):
    counts = df[col].value_counts(dropna=False)
    print(f"{len(counts)} distinct values")
    print(counts.to_string())

colValues("WaterfrontYN")

2 distinct values
WaterfrontYN
NaN     286965
True       162


In [157]:
df["WaterfrontYN"] = df["WaterfrontYN"].eq(True).astype(int)
df["BasementYN"] = df["BasementYN"].eq(True).astype(int)

For LotSize, avg sqft col and acres (in sqft) col, if one is null take the other measurement. If both are null flag in the missing column and set to 0. 

In [158]:
sqft = df["LotSizeSquareFeet"]
acres_sqft = df["LotSizeAcres"] * 43560
lot = (sqft.fillna(acres_sqft) + acres_sqft.fillna(sqft)) / 2

df["LotSizeSquareFeet_missing"] = lot.isna().astype(int)
df["LotSizeSquareFeet"] = lot.fillna(0)
df = df.drop(columns=["LotSizeAcres"])

For other YN columns, 1 for True, 0 for False, and 0 for null and flag in a col_missing column where 0 is not missing and 1 is missing

Flag other columns missing that have high null percentages

In [159]:
yn_cols = [
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
]

for col in yn_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].map({True: 1, False: 0}).fillna(0).astype(int)

flag_cols = [
    "GarageSpaces",
    "Levels",
    "Stories",
    "AssociationFee",
    "AssociationFeeFrequency",
    "BuildingAreaTotal",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "ParkingTotal",
]

for col in flag_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(0)

df.head(2)

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,PropertyType,PropertySubType,WaterfrontYN,...,Levels_missing,Stories_missing,AssociationFee_missing,AssociationFeeFrequency_missing,BuildingAreaTotal_missing,LivingArea_missing,BedroomsTotal_missing,BathroomsTotalInteger_missing,YearBuilt_missing,ParkingTotal_missing
0,35000.0,0.0,0.0,0.0,108900.0,0.0,0.0,Land,NaN,0,...,1,1,0,1,1,1,1,1,1,1
1,90000.0,0.0,0.0,0.0,653400.0,0.0,0.0,Land,NaN,0,...,1,1,1,1,1,1,1,1,1,1


One hot encoding on categorical columns
School District Column has cols for all unique districts with over 10 rows associated, Postal Codes and Cities are categorized within the top 50 and 100 respectively with the rest being put in other. Null values go in missing column. 

In [160]:
df["PostalCode"] = df["PostalCode"].astype("string").str.replace(r"\.0$", "", regex=True)

train = df["split"] == "train"
top_cities = df.loc[train, "City"].value_counts().head(50).index
top_zips = df.loc[train, "PostalCode"].value_counts().head(100).index
keep_districts = df.loc[train, "HighSchoolDistrict"].value_counts()
keep_districts = keep_districts[keep_districts > 10].index

df["City"] = df["City"].where(df["City"].isna() | df["City"].isin(top_cities), "Other")
df["PostalCode"] = df["PostalCode"].where(df["PostalCode"].isna() | df["PostalCode"].isin(top_zips), "Other")
df["HighSchoolDistrict"] = df["HighSchoolDistrict"].where(
    df["HighSchoolDistrict"].isna() | df["HighSchoolDistrict"].isin(keep_districts), "Other"
)

materials = sorted({part.strip() for value in df["Flooring"].dropna() for part in str(value).split(",") if part.strip()})
flooring_missing = df["Flooring"].isna()
flooring = df["Flooring"].fillna("")
for material in materials:
    df[f"Flooring_{material}"] = flooring.str.contains(rf"(?:^|,){material}(?:,|$)", regex=True).astype("int8")
df["Flooring_missing"] = flooring_missing.astype("int8")
df = df.drop(columns=["Flooring"])

cat_cols = ["City", "PostalCode", "HighSchoolDistrict", "PropertySubType"]
df = pd.get_dummies(df, columns=cat_cols, dummy_na=True, dtype="int8")
df.shape

(287127, 604)